<a href="https://colab.research.google.com/github/Syomara/CodeAlpha_Data-Analytics-Intership/blob/main/Task_1_CodeAlpha_Intership.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **CodeAlpha Intership - Data Analytics**
## Task 1:

*English:*

Objective:

- Perform web scraping
- Extract relevant information in a dataframe

*Español:*

Objetivo:

- Realizar web scraping
- Extraer información relevante en un dataframe

## Data Source/Fuente de datos

*English:*

For this project, the website of Escaramuza (https://escaramuza.com.uy/), a Uruguayan bookstore with an e-commerce platform, was chosen as the source for data extraction using web scraping.

This analysis is being conducted solely for educational purposes, as part of the Data Analytics internship at CodeAlpha. There are no plans to use the information obtained for commercial purposes or to redistribute the complete dataset.

The website's robots.txt file (https://escaramuza.com.uy/robots.txt) was previously checked, confirming that there are no access restrictions for bots (Allow: /).

*Español:*

Para realizar este trabajo se elige la página web de Escaramuza
(https://escaramuza.com.uy/), una librería uruguaya con e-commerce,
como fuente para la extracción de datos mediante web scraping.

Este análisis se realiza únicamente con fines educativos, en el marco
de la pasantía de Data Analytics de CodeAlpha. No se planea utilizar
la información obtenida con fines comerciales ni redistribuir el
dataset completo.

Se verificó previamente el archivo robots.txt del sitio
(https://escaramuza.com.uy/robots.txt), confirmando que no existen
restricciones de acceso para bots (`Allow: /`).

*English:*

This task involves performing web scraping, for which the BeautifulSoup library is used, which allows obtaining information from files in HTML format.

*Español:*

Esta tarea implica la realización de web scrapping, para esto se decide utilizar la biblioteca BeautifulSoup que permite obtener información de archivos en formato HTML.

In [4]:
#Library installation/Instalación de Biblioteca
!pip install beautifulsoup4

In [5]:
#The library is imported/Se importa la biblioteca
import requests
from bs4 import BeautifulSoup

In [6]:
#The content of the chosen URL is requested/Se solicita el contenido de la URL elegida
url = "https://escaramuza.com.uy/libros"
respuesta = requests.get(url)
print(respuesta.status_code)

200


*English:*

The request return is 200, this indicates that it is possible to deliver the information and work can continue.

*Español:*

La devolución de request es 200, esto indica que es posible entregar la información y se puede continuar trabajando.

In [7]:
#We proceed to verify if the information is in accordance with what we need/Procedemos a verificar si la información es acorde a lo que necesitamos
#We now proceed to interpret what was obtained in the previous line/Pasamos a interpretar lo que se obtuvo en la linea anterior
soup = BeautifulSoup(respuesta.text, "html.parser")
#The initial data is displayed to verify that useful information was obtained./Se muestran los primeros datos para verificar que se trajo información útil
print(soup.prettify()[:1000])

<!DOCTYPE html>
<html lang="es">
 <head>
  <!-- Primary Meta Tags -->
  <title>
   Libros - Escaramuza - Libros y café
  </title>
  <meta content="Libros - Escaramuza - Libros y café" name="title"/>
  <meta content="Somos una librería-café ubicada en Pablo de María y Charrúa, en el barrio Cordón de la ciudad de Montevideo. Podés venir a conocernos de lunes a sábados de 09:00 a 21:00." name="description"/>
  <link href="https://escaramuza.com.uy/libros/1" rel="canonical"/>
  <!-- Open Graph / Facebook -->
  <meta content="website" property="og:type"/>
  <!-- Open Graph Meta Tag -->
  <meta content="https://escaramuza.com.uy/libros" property="og:url"/>
  <meta content="Libros - Escaramuza - Libros y café" property="og:title"/>
  <meta content="Somos una librería-café ubicada en Pablo de María y Charrúa, en el barrio Cordón de la ciudad de Montevideo. Podés venir a conocernos de lunes a sábados de 09:00 a 21:00." property="og:description"/>
  <meta content="https://escaramuza.com.uy/files

*English:*

It's providing accurate information, so we've decided to continue analyzing and working with the different book categories. Since each book listing isn't labeled with a specific category, we'll first extract the categories and then extract the books by category.

To do this, we'll evaluate the HTML and search for the categories:

*Español:*

Está trayendo información correcta, se decide proseguir analizando y trabajando con las distintas categorías de libros ya que evaluando cada publicación de los libros se observa que no se denomina la categoría, por lo que primero vamos a extraer la categoría y luego vamos a extraer libros por categorías.

Para esto, se evalua el HTML y se buscan las categorías:

In [8]:
respuesta = requests.get("https://escaramuza.com.uy/")
soup = BeautifulSoup(respuesta.text, "html.parser")

menu_categorias = soup.find("div", class_="categoriesLevelThree")
categorias_links = menu_categorias.find_all("a", class_="categoryThreeTilte")

categorias = {}
for cat in categorias_links:
    nombre = cat.text.strip()
    href = cat.get("href")
    if nombre.lower() != "ver todas":  #The "view all" option, which includes the link to all books together, has been removed/Se elimina la opción ver todas que incluye el link a todos los libros juntos
        categorias[nombre] = f"https://escaramuza.com.uy{href[:-1]}"

print(categorias)

{'Arte': 'https://escaramuza.com.uy/libros/arte/', 'Ciencias sociales': 'https://escaramuza.com.uy/libros/ciencias-sociales/', 'Economía': 'https://escaramuza.com.uy/libros/economia/', 'Gastronomía': 'https://escaramuza.com.uy/libros/gastronomia/', 'Género': 'https://escaramuza.com.uy/libros/genero/', 'Infantiles': 'https://escaramuza.com.uy/libros/literatura-infantil/', 'Juveniles': 'https://escaramuza.com.uy/libros/literatura-juvenil/', 'Literatura': 'https://escaramuza.com.uy/libros/literatura/', 'Poesía': 'https://escaramuza.com.uy/libros/poesia/'}


*English:*

In addition to categories, there are pages that contain 24 books per page. Therefore, each page must be evaluated individually to retrieve all possible books. We will now evaluate each page within each category. To avoid overloading the page with requests, we must add a waiting time between queries. To do this, we import the `time` attribute.

*Español:*

Además de categorías, existen páginas de la busqueda que contienen 24 libros por página, por esto, se debe evaluar una a una las páginas para obtener todos los libros posibles. Procedemos a evaluar cada página de cada categoría. Para no saturar la página con los request debemos agregar tiempo de espera entre consulta y consulta, para esto se importa time.

In [9]:
import time

todos_los_libros = []

for nombre_categoria, base_url in categorias.items():
    for pagina in range(1, 26):
        url = f"{base_url}{pagina}"
        respuesta = requests.get(url)
        soup = BeautifulSoup(respuesta.text, "html.parser")

        libros = soup.find_all("a", class_="productViewContainer")

        for libro in libros:
            titulo_tag = libro.find("h2", class_="productViewName")
            titulo = titulo_tag.text.strip() if titulo_tag else None

            autor_tag = libro.find("h3", id="productAuthor")
            autor = autor_tag.text.strip() if autor_tag else None

            precio_tag = libro.find("div", class_="productViewPrice")
            precio = precio_tag.text.strip() if precio_tag else None

            disp_tag = libro.find("div", class_="productViewAditionalInfo")
            disponibilidad = disp_tag.text.strip() if disp_tag else None

            link = libro.get("href")

            todos_los_libros.append({
                "titulo": titulo,
                "autor": autor,
                "precio": precio,
                "disponibilidad": disponibilidad,
                "link": link,
                "categoria": nombre_categoria
            })

        print(f"{nombre_categoria} - página {pagina}: {len(libros)} libros (acumulado: {len(todos_los_libros)})")
        time.sleep(2)

print(f"\nTotal final: {len(todos_los_libros)} libros")

Arte - página 1: 24 libros (acumulado: 24)
Arte - página 2: 24 libros (acumulado: 48)
Arte - página 3: 24 libros (acumulado: 72)
Arte - página 4: 24 libros (acumulado: 96)
Arte - página 5: 24 libros (acumulado: 120)
Arte - página 6: 24 libros (acumulado: 144)
Arte - página 7: 24 libros (acumulado: 168)
Arte - página 8: 24 libros (acumulado: 192)
Arte - página 9: 24 libros (acumulado: 216)
Arte - página 10: 24 libros (acumulado: 240)
Arte - página 11: 24 libros (acumulado: 264)
Arte - página 12: 24 libros (acumulado: 288)
Arte - página 13: 24 libros (acumulado: 312)
Arte - página 14: 24 libros (acumulado: 336)
Arte - página 15: 24 libros (acumulado: 360)
Arte - página 16: 24 libros (acumulado: 384)
Arte - página 17: 24 libros (acumulado: 408)
Arte - página 18: 24 libros (acumulado: 432)
Arte - página 19: 10 libros (acumulado: 442)
Arte - página 20: 0 libros (acumulado: 442)
Arte - página 21: 0 libros (acumulado: 442)
Arte - página 22: 0 libros (acumulado: 442)
Arte - página 23: 0 libros

*English:*

We proceed to create the dataframe so that we can then proceed with the subsequent analysis.

*Español:*

Procedemos a crear el dataframe para luego poder proceder con el análisis posterior.

In [14]:
import pandas as pd

df = pd.DataFrame(todos_los_libros)

*English:*

We analyze the dimensions and shape of the created dataframe to verify that useful information has been created.

*Español:*

Analizamos las dimensiones y forma del dataframe creado para verificar que se haya creado información que luego sea útil.


In [11]:
print(df.shape)
df.head()

(4319, 6)


,titulo,autor,precio,disponibilidad,link,categoria
0,BLOCK IMANTADO 10X20 PRECISO COSAS,VARIOS AUTORES,"UYU 250,75",Disponible,/p/block-imantado-10x20-preciso-cosas/179654/2...,Arte
1,BLOCK IMANTADO 10X10 COLORBLOCK,VARIOS AUTORES,"UYU 250,75",Disponible,/p/block-imantado-10x10-colorblock/179653/283025,Arte
2,EL CAMINO DEL ARTISTA,JULIA CAMERON,"UYU 1.181,50",Disponible,/p/el-camino-del-artista/160123/111268,Arte
3,TECNICAS DE DIBUJO CON PLUMA Y TINTA,DAVID MORALES,UYU 935,Disponible,/p/tecnicas-de-dibujo-con-pluma-y-tinta/178740...,Arte
4,PINTURA DE PAISAJES,MITCHELL ALBALA,UYU 1.105,Disponible,/p/pintura-de-paisajes/173843/277710,Arte


*English:*

We will save it on GitHub in csv format to use it in the next task:

*Español:*

Lo guardaremos en GitHub en formato csv para luego utilizarlo en la siguiente tarea:

In [21]:
#We deleted it in case there was a file with that name/Borramos por si existe algún archivo con se nombre
!rm -rf CodeAlpha_Data-Analytics-Intership
#The repository to be used is cloned/Se clona el repositorio a usar
!git clone https://github.com/Syomara/CodeAlpha_Data-Analytics-Intership.git

Cloning into 'CodeAlpha_Data-Analytics-Intership'...
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 15 (delta 1), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (15/15), 177.48 KiB | 1.60 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [22]:
#We saved the file as a CSV file/Guardamos el archivo como csv
df.to_csv("/content/CodeAlpha_Data-Analytics-Intership/Task_1/escaramuza_libros.csv", index=False)

In [25]:
#We uploaded the file to GitHub/Subimos el archivo a GitHub
from google.colab import userdata
token = userdata.get('GitHub_token')

%cd /content/CodeAlpha_Data-Analytics-Intership/Task_1/
!git add escaramuza_libros.csv
!git commit -m "Agrego dataset de libros scrapeado de Escaramuza"
!git config --global user.email "syomarabazz@hotmail.com"
!git config --global user.name "Syomara"
!git push https://{token}@github.com/Syomara/CodeAlpha_Data-Analytics-Intership.git

/content/CodeAlpha_Data-Analytics-Intership/Task_1
[main 98001d7] Agrego dataset de libros scrapeado de Escaramuza
 1 file changed, 4325 insertions(+)
 create mode 100644 Task_1/escaramuza_libros.csv
Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 172.72 KiB | 8.22 MiB/s, done.
Total 4 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/Syomara/CodeAlpha_Data-Analytics-Intership.git
   500b131..98001d7  main -> main
